In [13]:
import numpy as np
import pandas as pd
from google.colab import files
df = pd.read_csv('/content/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv')
df

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...,...
26867,BL,I am waiting for a rebate of {{Refund Amount}}...,REFUND,track_refund,Thank you for sharing your situation regarding...
26868,BIL,how to see if there is anything wrong with my ...,REFUND,track_refund,Ensuring the accuracy of your restitution is o...
26869,BLQZ,I'm waiting for a reimbjrsement of {{Currency ...,REFUND,track_refund,"Firstly, I genuinely understand the importance..."
26870,BL,I don't know what to do to see my reimbursemen...,REFUND,track_refund,I've understood you're unsure about how to che...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  26872 non-null  object
 1   category     26872 non-null  object
 2   intent       26872 non-null  object
 3   response     26872 non-null  object
dtypes: object(4)
memory usage: 839.9+ KB


# 1. Detecting missing values

In [ ]:
df.isna().sum()
# No missing values

,0
instruction,0
category,0
intent,0
response,0


# 2. Removing duplicates

In [ ]:
df.duplicated().sum()
# No duplicates

np.int64(0)

## 3. Removing the flags column
The values in this column appear to some codes but no explanation is provided for what they represent. Since there is no way to find a meaningful interpretation of them, we decided to remove this column.

In [ ]:
df = df.drop(columns = ['flags'])
df

,instruction,category,intent,response
0,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...
26867,I am waiting for a rebate of {{Refund Amount}}...,REFUND,track_refund,Thank you for sharing your situation regarding...
26868,how to see if there is anything wrong with my ...,REFUND,track_refund,Ensuring the accuracy of your restitution is o...
26869,I'm waiting for a reimbjrsement of {{Currency ...,REFUND,track_refund,"Firstly, I genuinely understand the importance..."
26870,I don't know what to do to see my reimbursemen...,REFUND,track_refund,I've understood you're unsure about how to che...


# 4. Resolving One-to-Many Response Mappings.
During the data understanding phase, we observed that a single (instruction, category, intent) tuple could be associated with multiple semantically equivalent responses. Since the objective of this project is response retrieval rather than response generation, maintaining multiple paraphrased responses for the same input was unnecessary. Therefore, we standardized the dataset by retaining only the first response for each unique (instruction, category, intent) combination, creating a deterministic one-to-one mapping between each input tuple and its corresponding response.

In [ ]:
df = (
    df.drop_duplicates(
        subset=["instruction", "category", "intent"],
        keep="first"
    )
    .reset_index(drop=True)
)
response_counts = (
    df.groupby("instruction")["response"]
      .nunique()
      .sort_values(ascending=False)
)
response_counts

,response
instruction,
"you arenot being helpful at all, I want to speak to a person",1
ETA of oorder {{Order Number}},1
ETA of order {{Order Number}},1
ETA of ordr {{Order Number}},1
ETA of purchase {{Order Number}},1
...,...
I need assistance reporting an issue with online payment,1
I need assistance closing my platinum account,1
I need assistance checking the payment methods,1


# 5. Saving the file

In [17]:
df.to_csv('Bitext_Sample_Customer_Support_Training_Dataset.csv', index = False)
files.download('Bitext_Sample_Customer_Support_Training_Dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>